<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week8/Day4/ExerciseXP/Exercises_MCP_LLM_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises (Student) - MCP Client with LLM

In [2]:
!pip install -q mcp nest_asyncio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00


In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [3]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

# TODO (optional exercise): add multiply(a, b) here if you want an extra tool

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Writing server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

STDIO transport is generally simpler for local MCP development compared to HTTP because:

1.  **No Network Configuration:** It doesn't require setting up network ports, IP addresses, or dealing with firewall rules. Communication happens directly via standard input/output streams within the same machine.
2.  **No HTTP Server Overhead:** You don't need to run a dedicated HTTP server process, simplifying the deployment and reducing resource consumption for local testing.
3.  **Easier Debugging:** Debugging can be more straightforward as you're dealing with local process communication, often allowing direct inspection of stdin/stdout. HTTP involves more layers like request/response headers, status codes, and potential network issues.
4.  **Local Process Spawning:** For local development, it's often easier to simply spawn a child process and communicate with it via its standard I/O, which is exactly what STDIO transport facilitates.

## Exercise 2

In [1]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command=["python", "server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

ModuleNotFoundError: No module named 'mcp'

In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [4]:
async def ex3_list():
    params = StdioServerParameters(command=["python", "server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

La conversion d'une fonction Python standard en un "outil LLM" dans le contexte du cadre MCP (Multi-Component Protocol) implique la décoration de la fonction avec `@mcp.tool()`. Voici comment cela fonctionne en détail :

1.  **Décorateur `@mcp.tool()`** :
    *   Lorsque vous décorez une fonction comme `add` ou `greet` avec `@mcp.tool()`, vous indiquez au serveur MCP que cette fonction doit être exposée en tant qu'outil. Le décorateur s'occupe automatiquement de la majeure partie du travail pour rendre la fonction compatible avec le protocole MCP.

2.  **Inférer le schéma d'entrée (`inputSchema`)** :
    *   MCP utilise l'introspection de la signature de la fonction Python (les types des arguments et leurs annotations) pour générer un schéma JSON (JSON Schema) pour les entrées de l'outil. Ce schéma décrit les paramètres que l'outil attend, leurs types (`int`, `str`, etc.) et si un paramètre est requis ou facultatif.
    *   Par exemple, pour `add(a: int, b: int) -> int`, MCP déduit que l'outil `add` attend deux arguments de type entier, `a` et `b`. Le champ `properties` du schéma d'entrée reflétera cela : `{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}`.

3.  **Extraire le nom et la description de l'outil** :
    *   Le nom de l'outil est généralement tiré du nom de la fonction Python (par exemple, `add`, `greet`).
    *   La description de l'outil est extraite de la docstring de la fonction. C'est crucial car les modèles de langage (LLM) utilisent cette description pour comprendre la fonction de l'outil et décider quand l'appeler. Si aucune docstring n'est fournie, une description par défaut comme "mcp tool" est utilisée.

4.  **Formatage pour les LLM** :
    *   Les LLM modernes (comme Gemini, GPT, etc.) attendent les définitions d'outils dans un format spécifique, généralement un objet JSON avec des clés comme `type`, `function`, `name`, `description`, et `parameters` (qui inclut le `inputSchema` généré).
    *   La fonction `convert_to_llm_tool` que nous avons définie plus tôt prend l'objet `Tool` (tel qu'il est retourné par `session.list_tools()`) et le transforme en ce format spécifique que le LLM peut interpréter.

En résumé, `@mcp.tool()` permet de déclarer une fonction Python comme un outil, et MCP s'occupe de générer automatiquement un schéma d'entrée et une description que les LLM peuvent ensuite utiliser pour comprendre et invoquer l'outil via une interface standardisée.

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = #To-Do: convert tools to llm tool format
            calls = #To-Do: get tool calls from call_llm
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun